<a href="https://colab.research.google.com/github/doralalam/llm-engineering/blob/notes/003_week/day_3/029_tokenizers_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Tokenizers API**

In [1]:
# upgrade if exists, otherwise install

!pip install -q --upgrade datasets==3.6.0 transformers==4.57.6

In [1]:
# GPU validation

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed')>=0:
  print('Not connected to GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4')>=0:
    print('Success - connected T4 GPU')
  else:
    print('Not connected to T4 GPU')

Thu Aug 13 15:14:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# restart the kernel

import IPython

IPython.Application.instance().kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

In [2]:
# imports

from huggingface_hub import login
from google.colab import userdata

In [10]:
# hf_token validation

hf_token = userdata.get('HF_TOKEN')
if hf_token:
  print(f'HF_Key found and starts with {hf_token[0:3]}')
else:
  print('HF Key not found')

HF_Key found and starts with hf_


In [11]:
# Authenticate Python environment with HuggingFace

login(hf_token, add_to_git_credential=True)

# add_to_git_credential=True saves huggingface token into operation system's native Git credential manager
# this helps while uploading or importing the models or dataset from huggingface to Git using terminal without prompting for authentication

In [12]:
# imports

from transformers import AutoTokenizer

In [13]:
# initiating tokenizer using llama-3.1-8B model tokenizer

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B", trust_remote_code=True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

In [14]:
# converting the text into token_ids

text = "I am excited to see how Tokenizers work"
tokens = tokenizer.encode(text)
print(tokens)

[128000, 40, 1097, 12304, 311, 1518, 1268, 9857, 12509, 990]


In [17]:
char_count = len(text)
word_count = len(text.split(' '))
token_count = len(tokens)
print(f'There are {char_count} characters, {word_count} words and {token_count} tokens')
print(f'On an average, there will 1 token for 3/4 word')

There are 39 characters, 8 words and 10 tokens
On an average, there will 1 token for 3/4 word


In [18]:
# converting the token_ids back to words

print(tokenizer.decode(tokens))

<|begin_of_text|>I am excited to see how Tokenizers work


In [19]:
# converting batch of token_ids into list of words

print(tokenizer.batch_decode(tokens))

['<|begin_of_text|>', 'I', ' am', ' excited', ' to', ' see', ' how', ' Token', 'izers', ' work']


#### **Difference between `decode` and `batch_decode`**

`decode`

- converts 1D-tensors (single sequence) into single string

`batch_decode`

- converts 2D-tensors (batch of sequences) into list of strings

In [20]:
# to get the special tokens like <|begin_of_text|>, <|end_of_text|> etc.,

tokenizer.get_added_vocab()

{'<|begin_of_text|>': 128000,
 '<|end_of_text|>': 128001,
 '<|reserved_special_token_0|>': 128002,
 '<|reserved_special_token_1|>': 128003,
 '<|finetune_right_pad_id|>': 128004,
 '<|reserved_special_token_2|>': 128005,
 '<|start_header_id|>': 128006,
 '<|end_header_id|>': 128007,
 '<|eom_id|>': 128008,
 '<|eot_id|>': 128009,
 '<|python_tag|>': 128010,
 '<|reserved_special_token_3|>': 128011,
 '<|reserved_special_token_4|>': 128012,
 '<|reserved_special_token_5|>': 128013,
 '<|reserved_special_token_6|>': 128014,
 '<|reserved_special_token_7|>': 128015,
 '<|reserved_special_token_8|>': 128016,
 '<|reserved_special_token_9|>': 128017,
 '<|reserved_special_token_10|>': 128018,
 '<|reserved_special_token_11|>': 128019,
 '<|reserved_special_token_12|>': 128020,
 '<|reserved_special_token_13|>': 128021,
 '<|reserved_special_token_14|>': 128022,
 '<|reserved_special_token_15|>': 128023,
 '<|reserved_special_token_16|>': 128024,
 '<|reserved_special_token_17|>': 128025,
 '<|reserved_special_to

In [23]:
# to get the total no.of tokens availabe in the tokenizer

vocab_count = len(tokenizer.vocab)
print(f'Total number of token_ids available in llama-3.1-8B tokenizer is {vocab_count}')

Total number of token_ids available in llama-3.1-8B tokenizer is 128256


#### Instruct Models

- Many models have a variant that has been trained for use in chats

- Those models are typically labeled as instruct at the end

- They have been trained to expect prompts with a particular format that includes system, user, assistant etc.,

- There is a utility method `apply_chat_template` that will convert from the messages list format where we are familiar with, into the right input prompt for the model

In [24]:
# Initiating the tokenizer for chat variant model

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct", trust_remote_code=True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [25]:
# code to format a list of conversational messages into a single text string

messages = [
    {'role':'system', 'content':'You are a helpful assistant'},
    {'role':'user', 'content':'Tell a light-hearted joke for a room of Data Scientists'}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

Tell a light-hearted joke for a room of Data Scientists<|eot_id|><|start_header_id|>assistant<|end_header_id|>




#### Breakdown

- `apply_chat_template` - converts the conversational messages into a single text string by adding the special tokens wherever needed

- `tokenize=False` - returns the human readable text string instead of token_ids

- `tokenize=True` - returns the token ids

- `add_generation_prompt=True` - appends the model specific **assistant start tag** to the very end of the prompt. This signals to the LLM that it is now its turn to generate the assistant's response

#### Process

The messages to OpenAI gets converted to:

1. into a sequence of words with a special tags to separate the system, user, assistant

2. then the words broken down into fragments called token

3. tokens gets converted into numbers called token_ids that each token is assisned with a respective token_id

- `Input to an LLM is a sequence of token IDs and the output of the LLM is a probability distribution of the next token to follow this input`

#### Trying new models

In [26]:
PHI4 = "microsoft/Phi-4-mini-instruct"
DEEPSEEK = "deepseek-ai/DeepSeek-V3.1"
QWEN_CODER = "Qwen/Qwen2.5-Coder-7B-Instruct"

In [27]:
# Phi 4 tokenizer

phi4_tokenizer = AutoTokenizer.from_pretrained(PHI4)

text = "I am curiously excited to show Hugging Face Tokenizers in action to my LLM engineers"
print('Llama:')
tokens = tokenizer.encode(text)
print(tokens)
print(tokenizer.batch_decode(tokens))
print("\nPhi 4:")
tokens = phi4_tokenizer.encode(text)
print(tokens)
print(phi4_tokenizer.batch_decode(tokens))

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

Llama:
[128000, 40, 1097, 2917, 13610, 12304, 311, 1501, 473, 36368, 19109, 9857, 12509, 304, 1957, 311, 856, 445, 11237, 25175]
['<|begin_of_text|>', 'I', ' am', ' cur', 'iously', ' excited', ' to', ' show', ' H', 'ugging', ' Face', ' Token', 'izers', ' in', ' action', ' to', ' my', ' L', 'LM', ' engineers']

Phi 4:
[40, 939, 4396, 23138, 15209, 316, 2356, 59116, 4512, 29049, 17951, 24223, 306, 3736, 316, 922, 451, 19641, 32437]
['I', ' am', ' cur', 'iously', ' excited', ' to', ' show', ' Hug', 'ging', ' Face', ' Token', 'izers', ' in', ' action', ' to', ' my', ' L', 'LM', ' engineers']


In [30]:
# chat templates

print('Llama :')
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_promt=True))
print('-----------------------------')
print('Phi 4:')
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

Llama :
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

Tell a light-hearted joke for a room of Data Scientists<|eot_id|>
-----------------------------
Phi 4:
<|system|>You are a helpful assistant<|end|><|user|>Tell a light-hearted joke for a room of Data Scientists<|end|><|assistant|>


In [31]:
deepseek_tokenizer = AutoTokenizer.from_pretrained(DEEPSEEK)

text = "I am curiously excited to show Hugging Face Tokenizers in action to my LLM engineers"

print(tokenizer.encode(text))
print()
print(phi4_tokenizer.encode(text))
print()
print(deepseek_tokenizer.encode(text))

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[128000, 40, 1097, 2917, 13610, 12304, 311, 1501, 473, 36368, 19109, 9857, 12509, 304, 1957, 311, 856, 445, 11237, 25175]

[40, 939, 4396, 23138, 15209, 316, 2356, 59116, 4512, 29049, 17951, 24223, 306, 3736, 316, 922, 451, 19641, 32437]

[0, 43, 1030, 108771, 15046, 304, 1801, 24133, 5426, 11906, 47948, 24524, 295, 4271, 304, 1026, 33792, 47, 26170]


In [35]:
print('Llama: ')
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print('------------------------')
print('\nPhi 4: ')
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print('------------------------')
print('\nDeepSeek: ')
print(deepseek_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

Llama: 
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant<|eot_id|><|start_header_id|>user<|end_header_id|>

Tell a light-hearted joke for a room of Data Scientists<|eot_id|><|start_header_id|>assistant<|end_header_id|>


------------------------

Phi 4: 
<|system|>You are a helpful assistant<|end|><|user|>Tell a light-hearted joke for a room of Data Scientists<|end|><|assistant|>
------------------------

DeepSeek: 
<｜begin▁of▁sentence｜>You are a helpful assistant<｜User｜>Tell a light-hearted joke for a room of Data Scientists<｜Assistant｜></think>


In [36]:
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_CODER)
code = """
def hello_world(person):
  print("Hello", person)
"""
tokens = qwen_tokenizer.encode(code)
for token in tokens:
  print(f'{token} - {qwen_tokenizer.decode(token)}')

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

198 - 

750 - def
23811 -  hello
31792 - _world
29766 - (person
982 - ):

220 -  
1173 -  print
445 - ("
9707 - Hello
497 - ",
1697 -  person
340 - )

